# Evaluate the glyph segmentation detector

Run this locally (CPU is fine) after downloading `models/yolo_seg.pt`
from `notebooks/04_train_segmenter.ipynb`'s Colab run.

In [1]:
from pathlib import Path

from ultralytics import YOLO

CHECKPOINT_PATH = Path("..") / "models" / "yolo_seg.pt"

assert CHECKPOINT_PATH.exists(), (
    f"No checkpoint found at {CHECKPOINT_PATH} -- run notebooks/04_train_segmenter.ipynb "
    "on Colab first, download its output, and save it here as yolo_seg.pt."
)

## 1. Regenerate the held-out synthetic test split

Uses the same seeds as `notebooks/04_train_segmenter.ipynb` (`split_crop_paths(..., seed=0)`,
then `generate_dataset(test_crops, ..., seed=2)`), so this reproduces the
*exact same* test composites the training notebook held out -- no need to
download gigabytes of synthetic images from Colab, `data/raw/` is already
here locally.

In [2]:
from hieroglyph.segmentation.synthesize import generate_dataset, list_crop_paths, split_crop_paths

RAW_DIR = Path("..") / "data" / "raw"
SYNTH_DIR = Path("..") / "data" / "synthetic_composites"

crop_paths = list_crop_paths(RAW_DIR)
_, _, test_crops = split_crop_paths(crop_paths, seed=0)
generate_dataset(test_crops, SYNTH_DIR / "test", num_composites=500, seed=2)
print(f"{len(test_crops)} held-out test crops -> {SYNTH_DIR / 'test'}")

404 held-out test crops -> ..\data\synthetic_composites\test


In [3]:
import yaml

data_yaml = {
    "path": str(SYNTH_DIR.resolve()),
    "train": "test/images",  # unused by model.val(split="test"), but ultralytics requires the key
    "val": "test/images",
    "test": "test/images",
    "names": {0: "hieroglyph"},
    "nc": 1,
}
(SYNTH_DIR / "eval_data.yaml").write_text(yaml.dump(data_yaml), encoding="utf-8")

158

## 2. Quantitative: mAP on the held-out synthetic test set

In [4]:
model = YOLO(str(CHECKPOINT_PATH))
metrics = model.val(data=str(SYNTH_DIR / "eval_data.yaml"), split="test")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

Ultralytics 8.4.152  Python-3.13.12 torch-2.14.0+cpu CPU (Intel Core Ultra 7 155H)
YOLOv8n-seg summary (fused): 85 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 778.980.1 MB/s, size: 298.8 KB)
val: Scanning C:\Users\hbery\OneDrive\Desktop\hieroglyph\data\synthetic_composites\test\labels.cache... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 61.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 2.6s/it 1:222.6sss
                   all        500       3179      0.996      0.998      0.995      0.823      0.993      0.996      0.994      0.681
Speed: 2.7ms preprocess, 135.7ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to C:\Users\hbery\OneDrive\Desktop\hieroglyph\.claude\worktrees\glyph-segmentation-detector\runs\segment\val-3
mAP50: 0.9949
mAP50-95: 0.8230


## 3. Qualitative: real photos

Manual step first: download the "egyptian-hieroglyphs" dataset from
https://universe.roboflow.com/custom-yolov8-ljpde/egyptian-hieroglyphs
(Export -> any format, e.g. YOLOv8 -> download zip), unzip into
`../data/real_eval_photos/` so the next cell can find the images.

In [ ]:
import cv2

from hieroglyph.segmentation.classical import ClassicalSegmenter
from hieroglyph.segmentation.yolo import YoloSegmenter

REAL_EVAL_DIR = Path("..") / "data" / "real_eval_photos"
image_paths = sorted(REAL_EVAL_DIR.rglob("*.jpg")) + sorted(REAL_EVAL_DIR.rglob("*.png"))
assert image_paths, f"No real eval images found under {REAL_EVAL_DIR} -- see the download note above."

classical = ClassicalSegmenter()
trained = YoloSegmenter(CHECKPOINT_PATH)

for path in image_paths[:10]:
    image = cv2.imread(str(path))
    classical_boxes = classical.detect(image)
    trained_boxes = trained.detect(image)
    print(f"{path.name}: classical={len(classical_boxes)} boxes, trained={len(trained_boxes)} boxes")

## 4. Record the baseline

Copy the mAP50 number into the README (`## Model baseline` section, next
to the classifier's accuracy) once you've reviewed the qualitative
comparison above and are ready to treat this checkpoint as the one
`app/streamlit_app.py` builds on.